# Stage 6: Feature Diagnostics and Ablation

Compare identical linear-regression pipelines using temporal/history features (A) and A plus weather/rainfall (B). Entry and Exit remain separate. Preprocessing is fitted on training only; validation selects the model, and test is evaluated once. Current-hour observed weather is included, so results are retrospective unless prediction-hour weather forecasts are available operationally.

In [8]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

project_root = Path.cwd().resolve()
if project_root.name == 'Notebooks':
    project_root = project_root.parent
data_dir = project_root / 'DataProcessed'
target = 'ridership_count'
splits = {name: pd.read_csv(data_dir / f'metro_weather_{name}_prepared.csv', parse_dates=['time']) for name in ['train', 'validation', 'test']}
full_features = pd.read_csv(data_dir / 'metro_weather_features_final.csv', parse_dates=['time'])
for name, frame in splits.items():
    assert frame[['time', 'direction', target]].notna().all().all()
    assert not frame.duplicated(['time', 'direction']).any()
    assert set(frame.direction) == {'Entry', 'Exit'}
assert splits['train'].time.max() < splits['validation'].time.min() < splits['test'].time.min()
assert not full_features.duplicated(['time', 'direction']).any()
print('Rows:', {name: len(frame) for name, frame in splits.items()}, '| full diagnostic rows:', len(full_features))

Rows: {'train': 3780, 'validation': 810, 'test': 810} | full diagnostic rows: 7602


## Feature Diagnostics

Missingness, distributions, rainfall zero-inflation, and exact redundancies are summarized from the final feature file. Correlations used for screening are calculated on training data only. Equivalent or highly correlated features are excluded from both models as documented in the saved summary.

In [2]:
train = splits['train'].copy()
train_keys = train[['time', 'direction']]
train_diagnostics = full_features.merge(train_keys, on=['time', 'direction'], how='inner', validate='one_to_one')
diagnostic_rows = []
for column in full_features.columns:
    diagnostic_rows.append({'section': 'missingness', 'feature': column, 'detail': 'count', 'value': int(full_features[column].isna().sum())})
    diagnostic_rows.append({'section': 'missingness', 'feature': column, 'detail': 'percent', 'value': float(full_features[column].isna().mean() * 100)})
for column in full_features.select_dtypes(include='number').columns:
    stats = full_features[column].describe(percentiles=[0.25, 0.5, 0.75, 0.95])
    for stat in ['mean', 'std', 'min', '25%', '50%', '75%', '95%', 'max']:
        diagnostic_rows.append({'section': 'distribution', 'feature': column, 'detail': stat, 'value': float(stats[stat])})
rain_zero_pct = float(full_features['rain_mm'].eq(0).mean() * 100)
diagnostic_rows.append({'section': 'rainfall', 'feature': 'rain_mm', 'detail': 'zero_percent', 'value': rain_zero_pct})
diagnostic_rows.append({'section': 'rainfall', 'feature': 'rain_mm', 'detail': 'positive_hours', 'value': int(full_features['rain_mm'].gt(0).sum())})
train_corr = train_diagnostics.select_dtypes(include='number').corr()
for i, left in enumerate(train_corr.columns):
    for right in train_corr.columns[i + 1:]:
        corr = float(train_corr.loc[left, right])
        if abs(corr) >= 0.95:
            diagnostic_rows.append({'section': 'training_correlation', 'feature': left, 'detail': right, 'value': corr})
exclusions = {
    'time': 'Raw timestamp; audit/index only, not a predictor.',
    'businessday': 'Raw date identifier; redundant with temporal features.',
    'precipitation_mm': 'Exact duplicate of rain_mm.',
    'rainfall_intensity': 'Categorical derivation of rain_mm.',
    'is_raining': 'Binary derivation of rain_mm.',
    'hour': 'Represented by hour_sin/hour_cos.',
    'day_of_week': 'Represented by dow_sin/dow_cos.',
    'is_weekend': 'Redundant with day-of-week cycle for this encoding.',
    'apparent_temperature_c': 'Excluded from model B due to |training r| >= 0.95 with temperature_2m_c.',
    'ridership_rolling_mean_3h': 'Excluded from both models due to |training r| >= 0.95 with ridership_lag_2h; retain the exact lag.'
}
for feature, reason in exclusions.items():
    diagnostic_rows.append({'section': 'exclusion', 'feature': feature, 'detail': reason, 'value': np.nan})
diagnostics = pd.DataFrame(diagnostic_rows)
diagnostics_path = data_dir / 'feature_diagnostics_summary.csv'
diagnostics.to_csv(diagnostics_path, index=False)
assert full_features['rain_mm'].equals(full_features['precipitation_mm'])
assert full_features['is_raining'].equals(full_features['rain_mm'].gt(0).astype(int))
assert abs(train_corr.loc['temperature_2m_c', 'apparent_temperature_c']) >= 0.95
assert abs(train_corr.loc['ridership_lag_2h', 'ridership_rolling_mean_3h']) >= 0.95
print(f'Full file: {len(full_features):,} rows; missing values: {int(full_features.isna().sum().sum())}; duplicate keys: {int(full_features.duplicated(["time", "direction"]).sum())}')
print(f'Rainfall zeros: {rain_zero_pct:.2f}% ({int(full_features["rain_mm"].gt(0).sum()):,} positive observations).')
print('Training correlations: temp/apparent temp =', round(train_corr.loc['temperature_2m_c', 'apparent_temperature_c'], 3), '; lag2/rolling3 =', round(train_corr.loc['ridership_lag_2h', 'ridership_rolling_mean_3h'], 3))
print('Saved diagnostics:', diagnostics_path)

Full file: 7,602 rows; missing values: 5558; duplicate keys: 0
Rainfall zeros: 92.87% (542 positive observations).
Training correlations: temp/apparent temp = 0.959 ; lag2/rolling3 = 0.97
Saved diagnostics: /Users/shivaya/Desktop/Desktop - Utkarsh’s MacBook Pro/Research/DelhiMetroResearch/DataProcessed/feature_diagnostics_summary.csv


## Paired Ablation

Model A uses direction, temporal cycles/indicators, and historical ridership lags/means. Model B uses the identical A feature block plus selected weather and rainfall features. The model is linear regression in both cases; direction is one-hot encoded and numeric imputation/scaling is fit inside a training-only pipeline.

In [3]:
temporal_features = ['direction', 'month', 'is_peak_hour', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
historical_features = ['ridership_lag_1h', 'ridership_lag_2h', 'ridership_lag_3h', 'ridership_lag_24h', 'ridership_rolling_mean_6h', 'ridership_rolling_mean_24h']
weather_features = ['temperature_2m_c', 'relative_humidity_2m_pct', 'rain_mm', 'weather_code_wmo', 'surface_pressure_hpa', 'cloud_cover_pct', 'visibility_m', 'wind_speed_10m_kmh', 'rainfall_lag_1h', 'rainfall_lag_2h', 'rainfall_lag_3h', 'rainfall_sum_3h', 'rainfall_sum_6h', 'rainfall_sum_12h', 'rainfall_sum_24h']
features_A = temporal_features + historical_features
features_B = features_A + weather_features
assert set(features_B + [target, 'time']).issubset(train.columns)
assert target not in features_B and 'time' not in features_B
assert 'precipitation_mm' not in features_B and 'apparent_temperature_c' not in features_B
assert 'ridership_rolling_mean_3h' not in features_B

def make_model(features):
    numeric = [feature for feature in features if feature != 'direction']
    preprocessing = ColumnTransformer([
        ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric),
        ('direction', OneHotEncoder(handle_unknown='ignore'), ['direction'])
    ])
    return Pipeline([('preprocess', preprocessing), ('regression', LinearRegression())])

train = splits['train'].sort_values(['time', 'direction']).reset_index(drop=True)
validation = splits['validation'].sort_values(['time', 'direction']).reset_index(drop=True)
test = splits['test'].sort_values(['time', 'direction']).reset_index(drop=True)
assert train[['time', 'direction']].equals(train[['time', 'direction']].sort_values(['time', 'direction']).reset_index(drop=True))
assert train.time.max() < validation.time.min() < test.time.min()
models = {'A_temporal_history': (features_A, make_model(features_A)), 'B_plus_weather': (features_B, make_model(features_B))}
validation_predictions = validation[['time', 'direction', target]].copy()
for name, (features, model) in models.items():
    assert train[features + [target]].notna().all().all()
    assert validation[features + [target]].notna().all().all()
    model.fit(train[features], train[target])
    validation_predictions[name] = model.predict(validation[features])
assert validation_predictions[['time', 'direction', target]].notna().all().all()
assert not validation_predictions.duplicated(['time', 'direction']).any()
assert validation_predictions[['time', 'direction']].equals(validation[['time', 'direction']])

def score(actual, predicted):
    errors = np.asarray(predicted, dtype=float) - np.asarray(actual, dtype=float)
    actual = np.asarray(actual, dtype=float)
    nonzero = actual != 0
    mape = float(np.mean(np.abs(errors[nonzero] / actual[nonzero])) * 100) if nonzero.any() else np.nan
    return {'n': len(actual), 'n_nonzero_mape': int(nonzero.sum()), 'MAE': float(np.mean(np.abs(errors))), 'RMSE': float(np.sqrt(np.mean(errors ** 2))), 'MAPE_pct': mape}

validation_metrics = pd.DataFrame([{'model': name, **score(validation_predictions[target], validation_predictions[name])} for name in models])
selected_model = validation_metrics.sort_values(['MAE', 'RMSE']).iloc[0]['model']
assert validation_metrics.n.nunique() == 1
print('Validation (model selection by MAE):')
display(validation_metrics)
print('Validation-selected:', selected_model, '| identical evaluation keys:', len(validation_predictions))

Validation (model selection by MAE):


,model,n,n_nonzero_mape,MAE,RMSE,MAPE_pct
0,A_temporal_history,810,810,14930.636897,19941.441927,981.605613
1,B_plus_weather,810,810,15151.819070,19858.557215,886.176941


Validation-selected: A_temporal_history | identical evaluation keys: 810


In [6]:
test_predictions = test[['time', 'direction', target]].copy()
for name, (features, model) in models.items():
    test_predictions[name] = model.predict(test[features])
assert test_predictions[['time', 'direction', target]].notna().all().all()
assert not test_predictions.duplicated(['time', 'direction']).any()
assert test_predictions[['time', 'direction']].equals(test[['time', 'direction']])
test_metrics = pd.DataFrame([{'model': name, **score(test_predictions[target], test_predictions[name])} for name in models])
assert test_metrics.n.nunique() == 1 and test_metrics.n.iloc[0] == len(test_predictions)

metrics = pd.concat([validation_metrics.assign(split='validation'), test_metrics.assign(split='test')], ignore_index=True)
weather_delta = metrics.pivot(index='split', columns='model', values=['MAE', 'RMSE'])
for split_name in ['validation', 'test']:
    for metric in ['MAE', 'RMSE']:
        a = weather_delta.loc[split_name, (metric, 'A_temporal_history')]
        b = weather_delta.loc[split_name, (metric, 'B_plus_weather')]
        mask = metrics.split.eq(split_name) & metrics.model.eq('B_plus_weather')
        metrics.loc[mask, f'{metric}_change_vs_A'] = b - a
        metrics.loc[mask, f'{metric}_improvement_pct_vs_A'] = (a - b) / a * 100

predictions = pd.concat([
    validation_predictions.assign(split='validation'),
    test_predictions.assign(split='test')
], ignore_index=True).rename(columns={target: 'actual'}).melt(
    id_vars=['split', 'time', 'direction', 'actual'],
    var_name='model', value_name='prediction'
)
assert predictions.prediction.notna().all()
assert not predictions.duplicated(['split', 'model', 'time', 'direction']).any()
for split_name, expected in [('validation', validation), ('test', test)]:
    part = predictions[predictions.split.eq(split_name)]
    expected_actual = expected.set_index(['time', 'direction'])[target]
    key_sets = [set(map(tuple, group[['time', 'direction']].to_numpy())) for _, group in part.groupby('model')]
    assert len(key_sets) == 2 and key_sets[0] == key_sets[1] == set(map(tuple, expected[['time', 'direction']].to_numpy()))
    for name, group in part.groupby('model'):
        indexed = group.set_index(['time', 'direction'])
        assert np.array_equal(indexed['actual'].to_numpy(), expected_actual.reindex(indexed.index).to_numpy())
        recomputed = score(group.actual, group.prediction)
        saved_metric = metrics.loc[metrics.split.eq(split_name) & metrics.model.eq(name)].iloc[0]
        assert all(np.isclose(recomputed[key], saved_metric[key]) for key in ['MAE', 'RMSE', 'MAPE_pct'])
        assert recomputed['n'] == saved_metric['n'] and recomputed['n_nonzero_mape'] == saved_metric['n_nonzero_mape']

metrics_path = data_dir / 'feature_ablation_metrics.csv'
predictions_path = data_dir / 'feature_ablation_predictions.csv'
metrics.to_csv(metrics_path, index=False)
predictions.to_csv(predictions_path, index=False)
print('Test metrics (evaluated once after validation selection):')
display(metrics[metrics.split.eq('test')])
print('Weather increment: B minus A; negative MAE/RMSE change is improvement.')
print('Saved:', metrics_path, 'and', predictions_path)

Test metrics (evaluated once after validation selection):


,model,n,n_nonzero_mape,MAE,RMSE,MAPE_pct,split,MAE_change_vs_A,MAE_improvement_pct_vs_A,RMSE_change_vs_A,RMSE_improvement_pct_vs_A
2,A_temporal_history,810,810,15870.571792,21703.634552,884.926160,test,NaN,NaN,NaN,NaN
3,B_plus_weather,810,810,15861.764066,21788.450642,879.800092,test,-8.807726,0.055497,84.816091,-0.390792


Weather increment: B minus A; negative MAE/RMSE change is improvement.
Saved: /Users/shivaya/Desktop/Desktop - Utkarsh’s MacBook Pro/Research/DelhiMetroResearch/DataProcessed/feature_ablation_metrics.csv and /Users/shivaya/Desktop/Desktop - Utkarsh’s MacBook Pro/Research/DelhiMetroResearch/DataProcessed/feature_ablation_predictions.csv


In [7]:
a_val = metrics[(metrics.split == 'validation') & (metrics.model == 'A_temporal_history')].iloc[0]
b_val = metrics[(metrics.split == 'validation') & (metrics.model == 'B_plus_weather')].iloc[0]
a_test = metrics[(metrics.split == 'test') & (metrics.model == 'A_temporal_history')].iloc[0]
b_test = metrics[(metrics.split == 'test') & (metrics.model == 'B_plus_weather')].iloc[0]
print(f"Data: {len(full_features):,} source rows; {int(full_features.isna().sum().sum()):,} missing cells, concentrated in unavailable historical features; rainfall is zero in {rain_zero_pct:.1f}% of hours. Exact/derived and high-correlation features were excluded as documented.")
print(f"Validation A MAE/RMSE {a_val.MAE:.1f}/{a_val.RMSE:.1f}; B {b_val.MAE:.1f}/{b_val.RMSE:.1f}.")
print(f"Test A MAE/RMSE {a_test.MAE:.1f}/{a_test.RMSE:.1f}; B {b_test.MAE:.1f}/{b_test.RMSE:.1f}; weather MAE change {b_test.MAE_change_vs_A:.1f} ({b_test.MAE_improvement_pct_vs_A:.2f}%).")
print(f"Validation selected {selected_model}; test was used only for final comparison. Prediction-hour observed weather makes this a retrospective ablation, not operational unless forecasts are substituted.")
print('Next: test weather forecasts and direction-specific error stability in a strictly rolling-origin operational evaluation.')

Data: 7,602 source rows; 5,558 missing cells, concentrated in unavailable historical features; rainfall is zero in 92.9% of hours. Exact/derived and high-correlation features were excluded as documented.
Validation A MAE/RMSE 14930.6/19941.4; B 15151.8/19858.6.
Test A MAE/RMSE 15870.6/21703.6; B 15861.8/21788.5; weather MAE change -8.8 (0.06%).
Validation selected A_temporal_history; test was used only for final comparison. Prediction-hour observed weather makes this a retrospective ablation, not operational unless forecasts are substituted.
Next: test weather forecasts and direction-specific error stability in a strictly rolling-origin operational evaluation.


## Key Observations & Conclusion

- **Data quality:** 7,602 rows; zero duplicate time-direction keys. Historical-feature gaps account for missing values; rain is zero in 92.87% of observations. Exact duplicates and training-correlation redundancies were excluded.
- **Validation:** A MAE/RMSE/MAPE = 14,930.64 / 19,941.44 / 981.61%; B = 15,151.82 / 19,858.56 / 886.18%. Validation selected A by MAE.
- **Test:** A = 15,870.57 / 21,703.63 / 884.93%; B = 15,861.76 / 21,788.45 / 879.80%. Weather changed MAE by -8.81 (-0.06%) and RMSE by +84.82 (+0.39%): negligible incremental value.
- **Limitations and next step:** MAPE is secondary and inflated by small actual ridership. B uses observed prediction-hour weather, so results are retrospective; repeat with forecast weather for operational claims and examine direction-specific error stability.